In [3]:
!pip install -q pandas numpy scikit-learn nltk ipywidgets openpyxl
import pandas as pd
import numpy as np
import re
import nltk

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, clear_output

nltk.download('stopwords')

from nltk.corpus import stopwords

STOP_WORDS = set(stopwords.words('english'))

print("Upload your CSV or Excel dataset")

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

if file_name.endswith('.csv'):
    df = pd.read_csv(
        file_name,
        encoding='utf-8',
        on_bad_lines='skip'
    )
elif file_name.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(file_name)
else:
    raise ValueError("Please upload a CSV or Excel file.")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Upload your CSV or Excel dataset


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving news_sentiment_analysis.csv to news_sentiment_analysis.csv
Dataset loaded successfully!
Shape: (3500, 8)


,Source,Author,Title,Description,URL,Published At,Sentiment,Type
0,stgnews,Bridger Palmer,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",https://www.stgeorgeutah.com/news/archive/2024...,2024-07-12T23:45:25+00:00,positive,Business
1,Zimbabwe Mail,Staff Reporter,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",https://www.thezimbabwemail.com/business/busin...,2024-07-12T22:59:42+00:00,neutral,Business
2,4-traders,NaN,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,https://www.marketscreener.com/business-leader...,2024-07-12T22:52:55+00:00,positive,Business
3,4-traders,NaN,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,https://www.marketscreener.com/quote/stock/MCD...,2024-07-12T22:41:01+00:00,negative,Business
4,PLANET,NaN,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,https://www.npr.org/2024/07/12/1197961036/roof...,2024-07-12T22:28:19+00:00,positive,Business


In [4]:
print("Columns in your dataset:")

for i, column in enumerate(df.columns):
    print(i, ":", column)

Columns in your dataset:
0 : Source
1 : Author
2 : Title
3 : Description
4 : URL
5 : Published At
6 : Sentiment
7 : Type


In [5]:
# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Fill missing values
df['Title'] = df['Title'].fillna('')
df['Description'] = df['Description'].fillna('')

# Create the actual document used for Information Retrieval
df['Document'] = (
    df['Title'].astype(str) + ' ' +
    df['Description'].astype(str)
)

# Remove empty documents
df = df[
    df['Document'].str.strip() != ''
].reset_index(drop=True)

# Create document ID
df['Document_ID'] = range(1, len(df) + 1)

print("Total documents:", len(df))

display(
    df[
        [
            'Document_ID',
            'Title',
            'Description',
            'Type'
        ]
    ].head(10)
)

Total documents: 3500


,Document_ID,Title,Description,Type
0,1,Pine View High teacher wins Best in State awar...,"ST. GEORGE — Kaitlyn Larson, a first-year teac...",Business
1,2,Businesses Face Financial Strain Amid Liquidit...,"Harare, Zimbabwe – Local businesses are grappl...",Business
2,3,Musk donates to super pac working to elect Tru...,(marketscreener.com) Billionaire Elon Musk has...,Business
3,4,US FTC issues warning to franchisors over unfa...,(marketscreener.com) A U.S. trade regulator on...,Business
4,5,Rooftop solar's dark side,4.5 million households in the U.S. have solar ...,Business
5,6,Gabelli asks Paramount for details on National...,(marketscreener.com) Billionaire investor Mari...,Business
6,7,QWI INVESTMENTS : QWI) &ndash; ANNOUNCEMENT RE...,"(marketscreener.com) July 12, 2024 5:08 pm QWI...",Business
7,8,Rome Resources Announces Shareholder Approval ...,(marketscreener.com) Rome Resources Ltd. is pl...,Business
8,9,Fawcett accused of fronting scheme for hostile...,Business Reporter A POTENTIALLY bruising fight...,Business
9,10,What Makes Spynn Publicity The Top Choice For ...,With creative strategies and deep industry kno...,Business


In [6]:
def preprocess(text):

    text = str(text).lower()

    # Remove URLs
    text = re.sub(
        r'http\S+|www\S+',
        ' ',
        text
    )

    # Remove special characters and numbers
    text = re.sub(
        r'[^a-z\s]',
        ' ',
        text
    )

    # Tokenize
    words = text.split()

    # Remove stopwords
    words = [
        word
        for word in words
        if word not in STOP_WORDS
    ]

    return ' '.join(words)


df['Processed_Document'] = (
    df['Document'].apply(preprocess)
)

display(
    df[
        [
            'Document_ID',
            'Document',
            'Processed_Document'
        ]
    ].head()
)

,Document_ID,Document,Processed_Document
0,1,Pine View High teacher wins Best in State awar...,pine view high teacher wins best state award b...
1,2,Businesses Face Financial Strain Amid Liquidit...,businesses face financial strain amid liquidit...
2,3,Musk donates to super pac working to elect Tru...,musk donates super pac working elect trump blo...
3,4,US FTC issues warning to franchisors over unfa...,us ftc issues warning franchisors unfair busin...
4,5,Rooftop solar's dark side 4.5 million househol...,rooftop solar dark side million households u s...


In [7]:
vectorizer = TfidfVectorizer(
    lowercase=False,
    min_df=2,
    max_df=0.95
)

tfidf_matrix = vectorizer.fit_transform(
    df['Processed_Document']
)

terms = vectorizer.get_feature_names_out()

print("Number of documents:", tfidf_matrix.shape[0])
print("Number of terms:", tfidf_matrix.shape[1])
print("TF-IDF matrix shape:", tfidf_matrix.shape)

Number of documents: 3500
Number of terms: 11642
TF-IDF matrix shape: (3500, 11642)


In [8]:
# First 10 documents
# First 30 terms

tfidf_table = pd.DataFrame(
    tfidf_matrix[:10].toarray(),
    columns=terms
)

tfidf_table.insert(
    0,
    'Document_ID',
    df['Document_ID'].iloc[:10].values
)

display(
    tfidf_table.iloc[:, :31]
)

,Document_ID,aa,aacute,aan,aandacht,aaron,ab,abating,abbiamo,abbiamojesi,...,aboard,abortion,abr,abrandamento,abrdn,abroad,abscf,abschluss,absolute,absolutely
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
def search_documents(query, top_k=10):

    # -----------------------------------
    # PREPROCESS QUERY
    # -----------------------------------

    processed_query = preprocess(query)

    # Convert query to TF-IDF
    query_vector = vectorizer.transform(
        [processed_query]
    )

    # -----------------------------------
    # COSINE SIMILARITY
    # -----------------------------------

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # -----------------------------------
    # RANK DOCUMENTS
    # -----------------------------------

    ranked_indices = np.argsort(
        similarity_scores
    )[::-1][:top_k]

    # -----------------------------------
    # CREATE RESULTS
    # -----------------------------------

    results = df.iloc[
        ranked_indices
    ].copy()

    results['Similarity Score'] = (
        similarity_scores[ranked_indices]
    )

    results['Rank'] = range(
        1,
        len(results) + 1
    )

    # -----------------------------------
    # DISPLAY ONLY COLUMNS THAT EXIST
    # -----------------------------------

    desired_columns = [
        'Rank',
        'Document_ID',
        'Source',
        'Author',
        'Title',
        'Description',
        'Published',
        'Sentiment',
        'Type',
        'Similarity Score'
    ]

    # Keep only columns that actually exist
    available_columns = [
        col
        for col in desired_columns
        if col in results.columns
    ]

    results = results[available_columns]

    return results

In [15]:
query = "business market growth"

results = search_documents(
    query,
    top_k=10
)

display(results)

,Rank,Document_ID,Source,Author,Title,Description,Sentiment,Type,Similarity Score
2098,1,2099,4-traders,NaN,SpineGuard: growth accelerates in Q2,(marketscreener.com) SpineGuard lost nearly 5%...,positive,Technology,0.264724
1442,2,1443,neworleanscitybusiness,The Associated Press,Summer pause: Small business sales growth tape...,Small business sales growth slowed in June as ...,negative,Business,0.260185
2792,3,2793,finanznachrichten,NaN,Buy Now Pay Later Market: AI Integration Fuels...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,Technology,0.247862
3492,4,3493,finanznachrichten,NaN,Buy Now Pay Later Market: AI Integration Fuels...,"WESTFORD, Mass., July 18, 2024 /PRNewswire/ --...",neutral,Technology,0.247862
1540,5,1541,Financial Post | Canada Business News,Business Wire,Canada Ecommerce Market Databook 2024: 100+ KP...,DUBLIN &#8212; The &#8220;Canada Ecommerce Mar...,positive,Entertainment,0.234229
3429,6,3430,finanznachrichten,NaN,Research and Markets: Italy Ecommerce Databook...,"The ""Italy Ecommerce Market Opportunities Data...",neutral,Technology,0.221284
2210,7,2211,finanznachrichten,NaN,Research and Markets: Italy Ecommerce Databook...,"The ""Italy Ecommerce Market Opportunities Data...",neutral,Entertainment,0.221284
2729,8,2730,finanznachrichten,NaN,Research and Markets: Italy Ecommerce Databook...,"The ""Italy Ecommerce Market Opportunities Data...",neutral,Technology,0.221284
2910,9,2911,finanznachrichten,NaN,Research and Markets: Italy Ecommerce Databook...,"The ""Italy Ecommerce Market Opportunities Data...",neutral,Entertainment,0.221284
199,10,200,NBC,Davis Giangiulio,Redbox set to close as DVD market withers in s...,DVD rental service Redbox is set to shut down ...,negative,Entertainment,0.218634


In [16]:
def show_query_tfidf(query):

    processed_query = preprocess(query)

    query_vector = vectorizer.transform(
        [processed_query]
    )

    query_values = query_vector.toarray()[0]

    vocabulary = vectorizer.vocabulary_

    query_terms = [
        term
        for term in processed_query.split()
        if term in vocabulary
    ]

    if not query_terms:
        print("No query terms found in the vocabulary.")
        return

    query_table = pd.DataFrame({
        'Term': query_terms,
        'Query TF-IDF': [
            query_values[
                vocabulary[term]
            ]
            for term in query_terms
        ]
    })

    display(query_table)

In [17]:
show_query_tfidf(
    "business market growth"
)

,Term,Query TF-IDF
0,business,0.408290
1,market,0.640903
2,growth,0.650033


In [18]:
def show_document_tfidf(query, top_k=10):

    processed_query = preprocess(query)

    query_terms = processed_query.split()

    vocabulary = vectorizer.vocabulary_

    valid_terms = [
        term
        for term in query_terms
        if term in vocabulary
    ]

    if not valid_terms:
        print("No query terms found.")
        return

    # Get indexes of query terms
    term_indices = [
        vocabulary[term]
        for term in valid_terms
    ]

    # Get document TF-IDF values
    values = tfidf_matrix[
        :,
        term_indices
    ].toarray()

    table = pd.DataFrame(
        values,
        columns=valid_terms
    )

    table.insert(
        0,
        'Document_ID',
        df['Document_ID'].values
    )

    table.insert(
        1,
        'Title',
        df['Title'].values
    )

    # Sum of query-term TF-IDF values
    table['Total TF-IDF'] = table[
        valid_terms
    ].sum(axis=1)

    # Highest first
    table = table.sort_values(
        'Total TF-IDF',
        ascending=False
    )

    display(
        table.head(top_k)
    )

In [19]:
show_document_tfidf(
    "business market growth",
    top_k=10
)

,Document_ID,Title,business,market,growth,Total TF-IDF
1442,1443,Summer pause: Small business sales growth tape...,0.180284,0.000000,0.287027,0.467311
2098,2099,SpineGuard: growth accelerates in Q2,0.000000,0.000000,0.407247,0.407247
3492,3493,Buy Now Pay Later Market: AI Integration Fuels...,0.000000,0.255383,0.129511,0.384894
2792,2793,Buy Now Pay Later Market: AI Integration Fuels...,0.000000,0.255383,0.129511,0.384894
2873,2874,Research and Markets: Denmark Prepaid Card and...,0.112894,0.265818,0.000000,0.378712
2173,2174,Research and Markets: Denmark Prepaid Card and...,0.112894,0.265818,0.000000,0.378712
199,200,Redbox set to close as DVD market withers in s...,0.090331,0.283589,0.000000,0.373920
1540,1541,Canada Ecommerce Market Databook 2024: 100+ KP...,0.000000,0.365468,0.000000,0.365468
2210,2211,Research and Markets: Italy Ecommerce Databook...,0.000000,0.345269,0.000000,0.345269
2910,2911,Research and Markets: Italy Ecommerce Databook...,0.000000,0.345269,0.000000,0.345269


In [22]:
import ipywidgets as widgets
from IPython.display import display, clear_output


query_box = widgets.Text(
    placeholder='Enter your query here...',
    description='Query:',
    layout=widgets.Layout(width='80%')
)

top_k_box = widgets.IntSlider(
    value=10,
    min=1,
    max=20,
    step=1,
    description='Top K:'
)

search_button = widgets.Button(
    description='SEARCH',
    button_style='primary'
)

output = widgets.Output()


def perform_search(button):

    with output:

        clear_output()

        query = query_box.value.strip()

        if not query:
            print("Please enter a query.")
            return

        print("=" * 80)
        print("TF-IDF INFORMATION RETRIEVAL SYSTEM")
        print("=" * 80)

        print("\nUSER QUERY:")
        print(query)

        print("\nQUERY TF-IDF TABLE:")
        show_query_tfidf(query)

        print("\nRANKED DOCUMENTS:")

        results = search_documents(
            query,
            top_k=top_k_box.value
        )

        display(results)

        print("\nDOCUMENT TF-IDF TABLE:")

        show_document_tfidf(
            query,
            top_k=top_k_box.value
        )


search_button.on_click(perform_search)

display(
    query_box,
    top_k_box,
    search_button,
    output
)

Text(value='', description='Query:', layout=Layout(width='80%'), placeholder='Enter your query here...')

IntSlider(value=10, description='Top K:', max=20, min=1)

Button(button_style='primary', description='SEARCH', style=ButtonStyle())

Output()